# 准备 NinaPro 脉冲数据

本 Notebook 将 ninapro_data/spikes/T_{T}.zip 安全解压到 ninapro_data/spikes/T_{T}。

流程：

1. 检查 ZIP CRC、成员列表、路径穿越、符号链接、加密状态和未压缩大小；
2. 解压到同一文件系统中的临时目录；
3. 验证 metadata、NPZ 字段、shape、dtype、二值脉冲、计数与 Vin；
4. 验证通过后原子安装到目标目录。

默认不会覆盖已有的非空目标。确需替换时，在运行前设置环境变量 NINAPRO_OVERWRITE=1。


In [ ]:
# 1. 参数、依赖与路径
T = 40

import json
import os
import shutil
import stat
from hashlib import sha256
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

import numpy as np


def environment_flag(name, default=False):
    """读取常见形式的布尔环境变量。"""
    value = os.environ.get(name)
    if value is None:
        return default

    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"环境变量 {name} 必须是布尔值，实际为 {value!r}")


def find_project_root():
    """通过环境变量或当前目录向上寻找目标 ZIP。"""
    override = os.environ.get("NINAPRO_PROJECT_ROOT")
    start = Path(override).expanduser() if override else Path.cwd()
    start = start.resolve()
    archive_name = f"T_{T}.zip"

    for candidate in (start, *start.parents):
        archive_path = (
            candidate
            / "ninapro_data"
            / "spikes"
            / archive_name
        )
        if archive_path.is_file():
            return candidate

    raise FileNotFoundError(
        f"未找到 ninapro_data/spikes/{archive_name}。"
        "请从项目目录启动 Jupyter，或设置 NINAPRO_PROJECT_ROOT。"
    )


if isinstance(T, (bool, np.bool_)) or not isinstance(T, (int, np.integer)):
    raise TypeError("T 必须是正整数")
if T <= 0:
    raise ValueError("T 必须是正整数")
T = int(T)

PROJECT_ROOT = find_project_root()
SPIKE_ROOT = PROJECT_ROOT / "ninapro_data" / "spikes"
VERSION_NAME = f"T_{T}"
ARCHIVE_PATH = SPIKE_ROOT / f"{VERSION_NAME}.zip"
OUTPUT_ROOT = SPIKE_ROOT / VERSION_NAME
EXTRACT_ROOT = SPIKE_ROOT / f".{VERSION_NAME}-extracting"
OVERWRITE = environment_flag("NINAPRO_OVERWRITE", default=False)
MAX_UNCOMPRESSED_BYTES = int(
    float(os.environ.get("NINAPRO_MAX_EXTRACT_GIB", "10"))
    * 1024**3
)

REQUIRED_MEMBERS = {
    f"{VERSION_NAME}/metadata.json",
    f"{VERSION_NAME}/train.npz",
    f"{VERSION_NAME}/test.npz",
}
EXPECTED_NPZ_FIELDS = {
    "X",
    "spike_counts",
    "Vin",
    "y",
    "subject",
    "repetition",
    "start_sample",
}
SPLITS = ("train", "test")

print(f"项目目录：{PROJECT_ROOT}")
print(f"压缩包：{ARCHIVE_PATH}")
print(f"目标目录：{OUTPUT_ROOT}")
print(f"T：{T}")


In [ ]:
# 2. ZIP 安全检查与流式解压
def sha256_file(path, block_size=1024 * 1024):
    digest = sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_member(info):
    """拒绝绝对路径、父目录跳转、反斜杠、符号链接和加密成员。"""
    name = info.filename
    if "\\" in name:
        raise RuntimeError(f"ZIP 成员使用反斜杠：{name}")

    path = PurePosixPath(name)
    if path.is_absolute() or ".." in path.parts:
        raise RuntimeError(f"ZIP 成员路径不安全：{name}")
    if not path.parts or path.parts[0] != VERSION_NAME:
        raise RuntimeError(
            f"ZIP 成员不在 {VERSION_NAME} 目录：{name}"
        )
    if info.flag_bits & 0x1:
        raise RuntimeError(f"不支持加密 ZIP 成员：{name}")

    unix_mode = (info.external_attr >> 16) & 0xFFFF
    if unix_mode and stat.S_ISLNK(unix_mode):
        raise RuntimeError(f"ZIP 中不允许符号链接：{name}")
    return path


def inspect_archive(archive_path=ARCHIVE_PATH):
    """验证 ZIP 结构、CRC 和解压后大小上限。"""
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        raise FileNotFoundError(f"找不到压缩包：{archive_path}")

    with ZipFile(archive_path, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"ZIP CRC 校验失败：{bad_member}")

        file_infos = [
            info for info in archive.infolist() if not info.is_dir()
        ]
        paths = [validate_member(info) for info in file_infos]
        names = [path.as_posix() for path in paths]
        if len(names) != len(set(names)):
            raise RuntimeError("ZIP 中存在重复成员名称")

        member_names = set(names)
        if member_names != REQUIRED_MEMBERS:
            raise RuntimeError(
                "ZIP 成员不正确："
                f"缺失={sorted(REQUIRED_MEMBERS - member_names)}，"
                f"额外={sorted(member_names - REQUIRED_MEMBERS)}"
            )

        total_size = sum(info.file_size for info in file_infos)
        if total_size > MAX_UNCOMPRESSED_BYTES:
            raise RuntimeError(
                f"ZIP 解压后大小异常：{total_size / 1024**3:.2f} GiB，"
                f"上限为 {MAX_UNCOMPRESSED_BYTES / 1024**3:.2f} GiB"
            )

    return {
        "sha256": sha256_file(archive_path),
        "compressed_bytes": int(archive_path.stat().st_size),
        "uncompressed_bytes": int(total_size),
        "member_count": len(names),
    }


def validate_safe_spike_path(path):
    """限制删除和移动操作只能作用于 spikes 的子路径。"""
    path = Path(path).resolve()
    spike_root = SPIKE_ROOT.resolve()
    if path == spike_root or spike_root not in path.parents:
        raise ValueError(f"目标必须位于 {spike_root} 内：{path}")


def safe_extract(archive_path, destination):
    """逐成员复制，避免 ZipFile.extract 的路径语义差异。"""
    destination = Path(destination).resolve()
    validate_safe_spike_path(destination)
    destination.mkdir(parents=True, exist_ok=False)

    with ZipFile(archive_path, "r") as archive:
        for info in archive.infolist():
            if info.is_dir():
                continue

            relative_path = validate_member(info)
            target = destination.joinpath(*relative_path.parts).resolve()
            if destination not in target.parents:
                raise RuntimeError(
                    f"解压目标越界：{info.filename}"
                )

            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, "r") as source:
                with target.open("wb") as output:
                    shutil.copyfileobj(
                        source,
                        output,
                        length=1024 * 1024,
                    )


In [ ]:
# 3. 验证解压后的数据契约
def read_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def validate_spike_directory(spike_root):
    """验证 metadata 与 train/test NPZ 的完整契约。"""
    spike_root = Path(spike_root)
    metadata = read_json(spike_root / "metadata.json")

    if metadata.get("complete") is not True:
        raise RuntimeError("metadata 未标记完整")
    if metadata.get("dataset_scope") != "full_train_and_test":
        raise RuntimeError("压缩包不是完整训练集和测试集")
    if int(metadata.get("T", -1)) != T:
        raise RuntimeError(
            f"压缩包 T={metadata.get('T')}，当前 Notebook T={T}"
        )

    channel_count = 16
    window_samples = 40
    maximum_voltage = float(
        metadata["mapping"]["maximum_voltage"]
    )
    total_spikes = {}

    for split_name in SPLITS:
        expected_count = int(metadata["source_counts"][split_name])
        expected_shapes = metadata["output_shapes"][split_name]
        path = spike_root / f"{split_name}.npz"

        with np.load(path, allow_pickle=False) as loaded:
            if set(loaded.files) != EXPECTED_NPZ_FIELDS:
                raise RuntimeError(
                    f"{split_name}.npz 字段错误：{sorted(loaded.files)}"
                )

            binary = loaded["X"]
            counts = loaded["spike_counts"]
            vin = loaded["Vin"]
            y = loaded["y"]
            subject = loaded["subject"]
            repetition = loaded["repetition"]
            start_sample = loaded["start_sample"]

            expected_binary_shape = (expected_count, channel_count, T)
            expected_vin_shape = (
                expected_count,
                channel_count,
                window_samples,
            )
            if tuple(expected_shapes["X"]) != expected_binary_shape:
                raise RuntimeError(
                    f"{split_name} metadata X 形状错误"
                )
            if tuple(expected_shapes["spike_counts"]) != (
                expected_binary_shape
            ):
                raise RuntimeError(
                    f"{split_name} metadata spike_counts 形状错误"
                )
            if tuple(expected_shapes["Vin"]) != expected_vin_shape:
                raise RuntimeError(
                    f"{split_name} metadata Vin 形状错误"
                )
            if binary.shape != expected_binary_shape:
                raise RuntimeError(f"{split_name} X 形状错误")
            if counts.shape != expected_binary_shape:
                raise RuntimeError(
                    f"{split_name} spike_counts 形状错误"
                )
            if vin.shape != expected_vin_shape:
                raise RuntimeError(f"{split_name} Vin 形状错误")

            if binary.dtype != np.uint8:
                raise RuntimeError(f"{split_name} X 类型错误")
            if counts.dtype != np.uint16:
                raise RuntimeError(
                    f"{split_name} spike_counts 类型错误"
                )
            if vin.dtype != np.float32:
                raise RuntimeError(f"{split_name} Vin 类型错误")
            if y.dtype != np.int64:
                raise RuntimeError(f"{split_name} y 类型错误")
            if subject.dtype != np.int16:
                raise RuntimeError(f"{split_name} subject 类型错误")
            if repetition.dtype != np.int8:
                raise RuntimeError(
                    f"{split_name} repetition 类型错误"
                )
            if start_sample.dtype != np.int32:
                raise RuntimeError(
                    f"{split_name} start_sample 类型错误"
                )

            if not np.array_equal(
                binary,
                (counts > 0).astype(np.uint8),
            ):
                raise RuntimeError(
                    f"{split_name} 二值脉冲与计数不一致"
                )
            if not np.all(np.isfinite(vin)):
                raise RuntimeError(f"{split_name} Vin 包含非有限值")
            if np.any(vin < 0.0) or np.any(
                vin > maximum_voltage + 1e-6
            ):
                raise RuntimeError(f"{split_name} Vin 越界")

            for name, values in (
                ("y", y),
                ("subject", subject),
                ("repetition", repetition),
                ("start_sample", start_sample),
            ):
                if values.shape != (expected_count,):
                    raise RuntimeError(
                        f"{split_name} {name} 形状错误"
                    )

            if np.any(y < 0) or np.any(y > 11):
                raise RuntimeError(
                    f"{split_name} y 必须位于 0..11"
                )
            if np.any(subject < 1) or np.any(subject > 10):
                raise RuntimeError(
                    f"{split_name} subject 必须位于 1..10"
                )

            actual_class_counts = np.bincount(
                y,
                minlength=12,
            ).astype(int).tolist()
            if actual_class_counts != metadata["class_counts"][split_name]:
                raise RuntimeError(
                    f"{split_name} 类别数量与 metadata 不一致"
                )

            total_spikes[split_name] = int(
                np.sum(counts, dtype=np.uint64)
            )

    if total_spikes != metadata["total_spikes"]:
        raise RuntimeError(
            f"总脉冲数与 metadata 不一致：{total_spikes}"
        )
    return metadata


In [ ]:
# 4. 临时解压、验证并原子安装
def prepare_spike_data(overwrite=OVERWRITE):
    validate_safe_spike_path(OUTPUT_ROOT)
    validate_safe_spike_path(EXTRACT_ROOT)
    archive_info = inspect_archive(ARCHIVE_PATH)

    if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()) and not overwrite:
        raise FileExistsError(
            f"输出目录已经存在：{OUTPUT_ROOT}。"
            "如需重建，请设置 NINAPRO_OVERWRITE=1。"
        )

    if EXTRACT_ROOT.exists():
        if not overwrite:
            raise RuntimeError(
                f"存在上次未完成的临时目录：{EXTRACT_ROOT}。"
                "确认后设置 NINAPRO_OVERWRITE=1 重建。"
            )
        shutil.rmtree(EXTRACT_ROOT)

    try:
        safe_extract(ARCHIVE_PATH, EXTRACT_ROOT)
        extracted_spike = EXTRACT_ROOT / VERSION_NAME
        metadata = validate_spike_directory(extracted_spike)

        if OUTPUT_ROOT.exists():
            if any(OUTPUT_ROOT.iterdir()):
                shutil.rmtree(OUTPUT_ROOT)
            else:
                OUTPUT_ROOT.rmdir()

        extracted_spike.replace(OUTPUT_ROOT)
        EXTRACT_ROOT.rmdir()
    except Exception:
        if EXTRACT_ROOT.exists():
            shutil.rmtree(EXTRACT_ROOT)
        raise

    print(f"解压并验证完成：{OUTPUT_ROOT}")
    print(
        f"ZIP 大小：{archive_info['compressed_bytes'] / 1024**2:.2f} MiB"
    )
    print(
        f"解压后大小："
        f"{archive_info['uncompressed_bytes'] / 1024**2:.2f} MiB"
    )
    print(f"ZIP SHA-256：{archive_info['sha256']}")
    return {
        "archive": archive_info,
        "metadata": metadata,
    }


PREPARE_RESULT = prepare_spike_data(overwrite=OVERWRITE)
PREPARE_RESULT


In [ ]:
# 5. 最终快速确认
metadata = read_json(OUTPUT_ROOT / "metadata.json")
assert metadata["complete"] is True
assert metadata["dataset_scope"] == "full_train_and_test"
assert int(metadata["T"]) == T

for split_name in SPLITS:
    with np.load(
        OUTPUT_ROOT / f"{split_name}.npz",
        allow_pickle=False,
    ) as loaded:
        print(
            f"{split_name}: "
            f"X={loaded['X'].shape}/{loaded['X'].dtype}, "
            f"spike_counts="
            f"{loaded['spike_counts'].shape}/"
            f"{loaded['spike_counts'].dtype}, "
            f"Vin={loaded['Vin'].shape}/{loaded['Vin'].dtype}"
        )

print("NinaPro 脉冲数据准备完成。")
